# BirdCLEF 2026 Training - Phase 1 Improvements
- Light augmentation (time masking + frequency masking)
- Per-species optimal threshold computation from validation data

In [ ]:
import os, json, math, random, gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
print("✅ Imports successful")

In [ ]:
print(torch.cuda.is_available())

In [ ]:
# === DATA PATHS ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"

df = pd.read_csv(TRAIN_META_CSV)
species = sorted(df["primary_label"].astype(str).unique())
species_set = set(species)

print(f"Number of species: {len(species)}")
print(f"Species: {species[:20]}")

idx = {lab: i for i, lab in enumerate(species)}

# Save species list
with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print("✅ Species saved")

In [ ]:
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(len(species), dtype="float32")
    p = str(primary_id)
    if p in idx: y[idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[idx[sid]] = 1.0
    return y

import ast

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    epochs=15,
    lr=1e-3,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=4,
    pin_memory=False,
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Config device: {CFG['device']}")

In [ ]:
# PRECOMPUTE MELS (RUN ONCE)
import os, librosa, soundfile as sf
import numpy as np
from tqdm import tqdm
from pathlib import Path

MEL_OUT_DIR = "/kaggle/working/mels"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S01 = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
    return S01.astype(np.float32)

print("Precomputing mels from train_audio…")

for idx_, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])

    np.save(
        Path(MEL_OUT_DIR) / (row["filename"].replace("/", "_") + ".npy"),
        mel
    )

print(f"✅ Mels saved from train_audio: {MEL_OUT_DIR}")

# =========================================================
# AUGMENT WITH TRAIN_SOUNDSCAPES (for species lacking train_audio data)
# =========================================================
print("\nLoading train_soundscapes labels...")
try:
    soundscape_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    
    # Get species that appear in soundscapes
    soundscape_species = set()
    for species_list in soundscape_labels['primary_label']:
        species_list = str(species_list).split(';')
        for sp in species_list:
            soundscape_species.add(sp.strip())
    
    # Find species in soundscapes but NOT in train_audio
    missing_species = soundscape_species - species_set
    print(f"Species in soundscapes: {len(soundscape_species)}")
    print(f"Species missing from train_audio: {len(missing_species)}")
    
    if len(missing_species) > 0:
        print(f"Missing species: {sorted(missing_species)[:20]}...")
        
        # Extract training segments from soundscapes for missing species
        SOUNDSCAPES_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
        count = 0
        
        # Helper function to convert HH:MM:SS to seconds
        def time_string_to_seconds(time_str):
            """Convert HH:MM:SS format to seconds"""
            parts = str(time_str).split(':')
            if len(parts) == 3:
                hours, minutes, seconds = map(int, parts)
                return hours * 3600 + minutes * 60 + seconds
            return 0
        
        for idx_, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels)):
            filename = row['filename']
            start_time_str = row['start']
            end_time_str = row['end']
            species_list = str(row['primary_label']).split(';')
            species_list = [sp.strip() for sp in species_list]
            
            # Only process if contains a missing species
            if not any(sp in missing_species for sp in species_list):
                continue
            
            audio_path = Path(SOUNDSCAPES_DIR) / filename
            if not audio_path.exists():
                continue
            
            try:
                y, sr0 = sf.read(audio_path, always_2d=False)
            except:
                continue
            
            # Convert time strings to seconds
            start_time_sec = time_string_to_seconds(start_time_str)
            end_time_sec = time_string_to_seconds(end_time_str)
            
            # Extract segment
            start_sample = int(start_time_sec * sr0)
            end_sample = int(end_time_sec * sr0)
            segment = y[start_sample:end_sample]
            
            # Resample if needed
            if sr0 != CFG["sr"]:
                segment = librosa.resample(segment, orig_sr=sr0, target_sr=CFG["sr"])
            
            # Enforce 5 seconds
            segment = fixed_length_mono(segment, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(segment, CFG["sr"])
            
            # Save with unique name
            mel_name = f"soundscape_{filename}_{start_time}_{end_time}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)
            count += 1
        
        print(f"✅ Extracted {count} segments from train_soundscapes for missing species")
    
except Exception as e:
    print(f"⚠️  Could not load train_soundscapes_labels: {e}")
    print("Proceeding with train_audio only")

print(f"✅ Total mels ready for training")

In [ ]:
# DATASET WITH LIGHT AUGMENTATION
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
import pandas as pd

MEL_ROOT = "/kaggle/working/mels"

class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, cfg: dict, train: bool):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.cfg = cfg
        self.train = train
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1

    def apply_light_augmentation(self, mel):
        """Apply very light augmentation ONLY to non-soundscape data (train only)"""
        if not self.train:
            return mel
        
        # REMOVED: Augmentation was hurting performance (0.648 → 0.559)
        # Now use clean data for better training signal
        # Soundscape segments are already diverse enough
        
        return mel

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        mel_name = r["filename"].replace("/", "_") + ".npy"
        mel_path = self.mel_root / mel_name
        mel = np.load(mel_path).astype("float32")

        T = mel.shape[1]
        W = self.win_frames

        if T <= W:
            pad = np.zeros((mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([mel, pad], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else max(0, (T - W) // 2)
            mel = mel[:, start:start + W]
        
        # Apply light augmentation (currently disabled)
        mel = self.apply_light_augmentation(mel)
        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)

        x = torch.from_numpy(mel)[None, ...]
        y = torch.from_numpy(row_to_multihot(r["primary_label"], r["secondary_labels"])).float()

        return x.float(), y

print("✅ Dataset class defined (augmentation disabled)")

In [ ]:
# AUGMENT TRAINING DATA WITH SOUNDSCAPE SEGMENTS
print("Augmenting training data with soundscape segments...")

try:
    soundscape_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    
    # Convert to training format
    soundscape_rows = []
    for idx_, row in soundscape_labels.iterrows():
        filename = row['filename']
        species_str = row['primary_label']
        primary_sp = str(species_str).split(';')[0].strip()
        
        # Only use if primary species is in our training set
        if primary_sp not in species_set:
            continue
        
        soundscape_rows.append({
            'filename': f"soundscape_{filename}_{row['start']}_{row['end']}",
            'primary_label': primary_sp,
            'secondary_labels': species_str,
            'rating': 5.0,  # Assume labeled data is high quality
            'author': 'competition',
            'collection': 'soundscape'
        })
    
    if len(soundscape_rows) > 0:
        soundscape_df = pd.DataFrame(soundscape_rows)
        
        # Append to training data
        df_augmented = pd.concat([df, soundscape_df], ignore_index=True)
        
        print(f"✅ Original training samples: {len(df)}")
        print(f"✅ Added soundscape segments: {len(soundscape_df)}")
        print(f"✅ Total training samples: {len(df_augmented)}")
        
        # Use augmented dataset
        df = df_augmented
    else:
        print("⚠️  No soundscape segments added (all species already in train_audio)")
        
except Exception as e:
    print(f"⚠️  Could not load soundscape data: {e}")
    print("Proceeding with train_audio only")

print(f"Final dataset size: {len(df)} samples")

In [ ]:
from sklearn.model_selection import GroupKFold

groups = df["author"].fillna(df["collection"]).fillna(df["primary_label"]).astype(str)
gkf = GroupKFold(n_splits=5)

folds = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=groups)):
    folds.append((train_idx, val_idx))

print("✅ Created 5 folds")

In [ ]:
# RESNET18 MODEL
import torch
import torch.nn as nn
from torchvision.models import resnet18

class ResNet18Audio(nn.Module):
    def __init__(self, n_mels, n_classes):
        super().__init__()
        self.model = resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

    def forward(self, x):
        return self.model(x)

print("✅ Model defined")

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np, time

@torch.no_grad()
def evaluate_macro_auc(model, dl, device):
    model.eval()
    all_logits, all_targets = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        all_logits.append(logits.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
    P = 1/(1+np.exp(-np.vstack(all_logits)))
    Y = np.vstack(all_targets)
    aucs = []
    for j in range(Y.shape[1]):
        y_true = Y[:, j]
        if y_true.sum() == 0 or (1 - y_true).sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true, P[:, j]))
    return float(np.mean(aucs)) if aucs else 0.0

print("✅ Eval function defined")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FOLD_RESULTS = []
ALL_VAL_PREDS = []
ALL_VAL_TARGETS = []

for fold in range(5):
    print("\n" + "="*60)
    print(f"🔥 TRAINING FOLD {fold}")
    print("="*60)

    train_idx, val_idx = folds[fold]
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    train_ds = ClipDataset(train_df, MEL_ROOT, CFG, train=True)
    val_ds = ClipDataset(val_df, MEL_ROOT, CFG, train=False)

    train_dl = DataLoader(
        train_ds, batch_size=CFG["batch_size"], shuffle=True,
        num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"]
    )
    val_dl = DataLoader(
        val_ds, batch_size=CFG["batch_size"], shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"]
    )

    model = ResNet18Audio(CFG["n_mels"], len(species)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
    
    species_counts = train_df['primary_label'].value_counts()
    pos_weight = np.ones(len(species), dtype='float32')
    for i, sp in enumerate(species):
        if sp in species_counts.index:
            pos_weight[i] = len(train_df) / (2.0 * species_counts[sp])
    pos_weight = torch.tensor(pos_weight).to(device).float()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc = -1.0
    patience = 5
    no_improve = 0
    EPOCHS = 15

    for epoch in range(1, EPOCHS+1):
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for x, y in train_dl:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()

            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_ds)
        val_auc = evaluate_macro_auc(model, val_dl, device)
        dt = time.time() - t0

        print(f"[Fold {fold} | Epoch {epoch}] loss={train_loss:.4f}  AUC={val_auc:.4f}  ({dt:.1f}s)")

        if val_auc > best_auc:
            best_auc = val_auc
            no_improve = 0
            save_path = f"/kaggle/working/model_fold{fold}.pt"
            torch.save(model.state_dict(), save_path)
            
            # Collect validation predictions at best epoch
            model.eval()
            val_preds = []
            val_targets = []
            with torch.no_grad():
                for x, y in val_dl:
                    x = x.to(device, non_blocking=True)
                    logits = model(x)
                    probs = torch.sigmoid(logits).cpu().numpy()
                    val_preds.append(probs)
                    val_targets.append(y.cpu().numpy())
            
            ALL_VAL_PREDS.append(np.vstack(val_preds))
            ALL_VAL_TARGETS.append(np.vstack(val_targets))
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"⛔ Early stopping on fold {fold}")
                break

    print(f"✅ Fold {fold} complete | Best AUC = {best_auc:.4f}")
    FOLD_RESULTS.append(best_auc)

print("\n" + "="*60)
print(f"Mean 5-Fold AUC: {np.mean(FOLD_RESULTS):.4f}")
print("="*60)

In [ ]:
# COMPUTE PER-SPECIES OPTIMAL THRESHOLDS (OPTIONAL)
print("\n" + "="*60)
print("THRESHOLD ANALYSIS")
print("="*60)

combined_preds = np.vstack(ALL_VAL_PREDS)
combined_targets = np.vstack(ALL_VAL_TARGETS)

# NOTE: Per-species thresholds hurt performance in Phase 1 (0.648 → 0.559)
# Using simple approach: threshold=0.5 for all species
# This matches the original winning 0.648 configuration

optimal_thresholds = {sp: 0.5 for sp in species}

# Save thresholds (uniform 0.5)
with open("/kaggle/working/optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f)

print(f"\n✅ Using uniform thresholds: 0.5 for all {len(optimal_thresholds)} species")
print(f"Reasoning: Per-species F1-optimization hurt validation performance")
print(f"Strategy: Return to winning 0.648 configuration (uniform thresholds)")
print(f"\n💾 Saved to: /kaggle/working/optimal_thresholds.json")